# Benamou--Brenier duality in one dimension

This notebook generates `fig:dynamic-benamou-brenier-duality`.  In one dimension the optimal quadratic transport is the monotone rearrangement, so the Benamou--Brenier primal variables can be evaluated without solving a PDE.  If
$$
    x_0(u)=F_0^{-1}(u),\qquad x_1(u)=F_1^{-1}(u),\qquad z_t(u)=(1-t)x_0(u)+t x_1(u),
$$
then
$$
    \rho_t(z_t(u)) = \frac{1}{\partial_u z_t(u)},\qquad
    v_t(z_t(u)) = x_1(u)-x_0(u),\qquad
    m_t=\rho_t v_t.
$$
The dual Hamilton--Jacobi potential is built from
$$
    \partial_x \phi_0(x_0(u)) = 2(x_1(u)-x_0(u)),\qquad
    \phi_t(z_t(u)) = \phi_0(x_0(u)) + t |x_1(u)-x_0(u)|^2,
$$
which gives $\partial_t\phi_t+|\partial_x\phi_t|^2/4=0$ along the active transported mass and $m_t=(\rho_t/2)\partial_x\phi_t$.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
from figure_style import RED, BLUE, VIOLET, GRAY, setup_matplotlib, figure_dir, save_pdf, box_axes, interp_color
setup_matplotlib()

NAME = "dynamic-benamou-brenier-duality"
out = figure_dir(NAME)
thumb_dir = ROOT / "notebooks-figures" / "thumbnails"
thumb_dir.mkdir(parents=True, exist_ok=True)

# Smooth one-dimensional endpoint densities: a two-bump source and a three-bump target.
def gaussian(x, mean, sigma):
    return np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)

def mixture_density(x, components):
    y = np.zeros_like(x)
    for weight, mean, sigma in components:
        y += weight * gaussian(x, mean, sigma)
    return y

def normalize_density(x, rho):
    mass = np.trapezoid(rho, x)
    return rho / mass

def cumulative_from_density(x, rho):
    increments = 0.5 * (rho[:-1] + rho[1:]) * np.diff(x)
    cdf = np.concatenate([[0.0], np.cumsum(increments)])
    cdf /= cdf[-1]
    cdf[0] = 0.0
    cdf[-1] = 1.0
    return cdf

x_grid = np.linspace(-4.4, 4.4, 6000)
rho0_grid = normalize_density(x_grid, mixture_density(x_grid, [(0.62, -1.45, 0.32), (0.38, 0.15, 0.43)]))
rho1_grid = normalize_density(x_grid, mixture_density(x_grid, [(0.32, -0.60, 0.25), (0.28, 0.75, 0.21), (0.40, 1.75, 0.36)]))
F0 = cumulative_from_density(x_grid, rho0_grid)
F1 = cumulative_from_density(x_grid, rho1_grid)

# Quantile grid.  The endpoints are trimmed only to avoid numerical infinities in the far tails.
u = np.linspace(2e-4, 1 - 2e-4, 3600)
x0 = np.interp(u, F0, x_grid)
x1 = np.interp(u, F1, x_grid)
w = x1 - x0

# Construct the initial dual potential by integrating phi'_0=2(T-Id).
phi0 = np.zeros_like(x0)
phi0[1:] = np.cumsum((w[:-1] + w[1:]) * np.diff(x0))  # trapezoid for integral of 2w dx
phi0 -= np.trapezoid(phi0, u)  # harmless additive gauge

# Numerical primal-dual checks.
w2_value = np.trapezoid(w**2, u)
dual_value = np.trapezoid((phi0 + w**2) - phi0, u)
np.testing.assert_allclose(w2_value, dual_value, rtol=3e-4, atol=3e-4)
print(f"W2^2 primal value = {w2_value:.6f}; dual endpoint gap = {dual_value:.6f}")

times = [0.0, 0.25, 0.5, 0.75, 1.0]

# Evaluate density, momentum and dual potential along the monotone quantile parametrization.
def solution_at(t):
    z = (1 - t) * x0 + t * x1
    dz_du = np.gradient(z, u, edge_order=2)
    rho = 1.0 / np.maximum(dz_du, 1e-10)
    m = rho * w
    phi = phi0 + t * w**2
    dphi_dz = np.gradient(phi, z, edge_order=2)
    hj = -w**2 + 0.25 * dphi_dz**2
    active = rho > 1e-3 * rho.max()
    assert np.max(np.abs(hj[active])) < 2e-2, "Hamilton-Jacobi residual too large on active support"
    return z, rho, m, phi, dphi_dz

solutions = {t: solution_at(t) for t in times}

xlim = (-2.7, 2.8)

def finish_axis(ax, ylabel):
    ax.set_xlim(*xlim)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(ylabel)
    box_axes(ax)
    ax.grid(True, color="#e6e1d8", lw=0.45, alpha=0.75)
    ax.tick_params(labelsize=8)

fig, ax = plt.subplots(figsize=(4.2, 2.25))
for t in times:
    z, rho, _, _, _ = solutions[t]
    color = interp_color(t)
    ax.plot(z, rho, color=color, lw=1.55, label=fr"$t={t:g}$")
    ax.fill_between(z, 0, rho, color=color, alpha=0.07, linewidth=0)
finish_axis(ax, r"$\rho_t$")
ax.legend(frameon=False, fontsize=7, ncol=1, loc="upper left", handlelength=1.4)
save_pdf(fig, out / "density.pdf")
plt.close(fig)

fig, ax = plt.subplots(figsize=(4.2, 2.25))
for t in times:
    z, _, m, _, _ = solutions[t]
    color = interp_color(t)
    ax.plot(z, m, color=color, lw=1.45)
ax.axhline(0, color="#3a3a3a", lw=0.55)
finish_axis(ax, r"$m_t=\rho_t v_t$")
save_pdf(fig, out / "momentum.pdf")
plt.close(fig)

fig, ax = plt.subplots(figsize=(4.2, 2.25))
for t in times:
    z, _, _, phi, _ = solutions[t]
    color = interp_color(t)
    ax.plot(z, phi, color=color, lw=1.45)
finish_axis(ax, r"$\phi_t$")
save_pdf(fig, out / "dual-potential.pdf")
plt.close(fig)

# Thumbnail/contact sheet for the notebook gallery.
fig, axes = plt.subplots(1, 3, figsize=(8.9, 2.15), constrained_layout=True)
for t in times:
    z, rho, m, phi, _ = solutions[t]
    c = interp_color(t)
    axes[0].plot(z, rho, color=c, lw=1.3)
    axes[0].fill_between(z, 0, rho, color=c, alpha=0.06, linewidth=0)
    axes[1].plot(z, m, color=c, lw=1.2)
    axes[2].plot(z, phi, color=c, lw=1.2)
axes[1].axhline(0, color="#3a3a3a", lw=0.5)
for ax, label in zip(axes, [r"$\rho_t$", r"$m_t$", r"$\phi_t$"]):
    finish_axis(ax, label)
fig.savefig(thumb_dir / f"{NAME}.png", dpi=190, bbox_inches="tight", pad_inches=0.04)
plt.close(fig)

print(f"Wrote panels to {out}")
print(f"Wrote thumbnail to {thumb_dir / (NAME + '.png')}")


W2^2 primal value = 2.558337; dual endpoint gap = 2.558337


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


...

'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


Wrote panels to /Users/gpeyre/Dropbox/github/ot4ml/OT4ML/figures/dynamic-benamou-brenier-duality
Wrote thumbnail to /Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/dynamic-benamou-brenier-duality.png
